# Extracting real data values from McIDAS-V (not pixels)

In [ ]:
%load_ext mcidasv_jupyter
%mcv_connect /path/to/runMcV
%mcv_replay off

## 1. Pull the layer's actual values + navigation

In [ ]:
import mcidasv_jupyter as mcv
import numpy as np
session = mcv.get_session()

field = session.extract_field('''
data = loadADDEImage(server='adde.ucar.edu', dataset='EAST', descriptor='CONUSC13',
                     size='ALL', unit='TEMP', mag=(-4, -4))
panel = buildWindow(height=400, width=500)
layer = panel[0].createLayer('Image Display', data)
''')

print(field.shape, field.unit)
print('%.1f .. %.1f %s' % (np.nanmin(field.masked()), np.nanmax(field.masked()), field.unit))
print('lat %.2f..%.2f  lon %.2f..%.2f' % (np.nanmin(field.lats), np.nanmax(field.lats),
                                          np.nanmin(field.lons), np.nanmax(field.lons)))

## 2. Physical analysis in real units

In [ ]:
import matplotlib.pyplot as plt

tb = field.masked()
deep = tb < 220.0
print('deep convection pixels (<220 K):', int(np.nansum(deep)))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
im = ax[0].imshow(tb, cmap='inferno_r'); ax[0].set_title('brightness temperature (K)')
ax[0].axis('off'); fig.colorbar(im, ax=ax[0], fraction=0.046)
ax[1].hist(tb[np.isfinite(tb)].ravel(), bins=80, color='steelblue')
ax[1].axvline(220, color='r', ls='--'); ax[1].set_xlabel('K'); ax[1].set_title('histogram')
plt.tight_layout()

## 3. Locate the coldest tops by latitude/longitude

In [ ]:
v = np.where(np.isfinite(tb), tb, np.inf)
r, c = np.unravel_index(np.nanargmin(v), tb.shape)
print('coldest %.1f K at %.2f N, %.2f E' % (tb[r, c], field.lats[r, c], field.lons[r, c]))

rows, cols = np.where(deep & field.valid)
if len(rows):
    print('deep-convection centroid: %.2f N, %.2f E' % (field.lats[rows, cols].mean(), field.lons[rows, cols].mean()))

## 4. Regrid to lat/lon using the true navigation

In [ ]:
from scipy.interpolate import griddata

m = np.isfinite(tb) & field.valid
pts = np.column_stack([field.lats[m], field.lons[m]])
glats = np.linspace(50, 22, 180)
glons = np.linspace(-122, -68, 300)
GLA, GLO = np.meshgrid(glats, glons, indexing='ij')

grid = griddata(pts, tb[m], (GLA, GLO), method='linear')
grid = np.where(np.isfinite(grid), grid, 300.0).astype('f4')
print('regridded:', grid.shape)

## 5. Send the regridded field back to McIDAS-V

In [ ]:
session.run('''
panel = buildWindow(height=500, width=750)
layer = panel[0].createLayer('Color-Shaded Plan View', g)
panel[0].setProjection('US>CONUS')
panel[0].setWireframe(False)
layer.setEnhancement('ABI IR Temperature', range=(200, 300))
layer.setLayerLabel(label='brightness temperature regridded from getData()')
''', arrays={'g': (grid, glats, glons)})

## 6. Deep-convection mask, correctly geolocated

In [ ]:
mask = griddata(pts, (tb[m] < 220).astype('f4'), (GLA, GLO), method='nearest')
mask = np.nan_to_num(mask).astype('f4')

session.run('''
panel = buildWindow(height=500, width=750)
layer = panel[0].createLayer('Color-Shaded Plan View', g)
panel[0].setProjection('US>CONUS')
panel[0].setWireframe(False)
layer.setLayerLabel(label='deep convection (Tb < 220 K)')
''', arrays={'g': (mask, glats, glons)})